# Training Pipeline — ConvNeXt

The same pipeline as [`resnet50.ipynb`](resnet50.ipynb) and the
[template](../../../_templates/Evaluation_Template.ipynb), but the backbone is
**ConvNeXt** from Facebook Research's
[`facebookresearch/ConvNeXt`](https://github.com/facebookresearch/ConvNeXt)
(Liu et al., *A ConvNet for the 2020s*). Data, loss, optimizer, schedule, training loop and
metrics are untouched, so the run stays directly comparable with the other notebooks in this
folder.

**No wrapper needed, unlike [`regnet.ipynb`](regnet.ipynb).** [timm](https://timm.fast.ai/) carries
a PyTorch ConvNeXt implementation contributed with the paper authors' involvement, loading the
**same converted checkpoints** the `facebookresearch/ConvNeXt` repo publishes — so there is no
TensorFlow/config mismatch to bridge, unlike pycls' global-config RegNet builder. That means
`'convnext_tiny'` is built by plain `timm.create_model`, exactly like `'resnet50'` in the first
notebook: no second clone, no wrapper, `model_classes` stays empty.

The other **reusable pieces** (model selection, the training routine, the metric helpers and the
all-metrics evaluation) live in [`_handlers/evaluation.py`](../../../_handlers/evaluation.py); the
notebook keeps only the configuration and the linear flow.

Use a GPU (a Colab **T4** is enough) — `build_model` calls `.cuda()` on the network.

## Install Requirements

Only what the pipeline actually imports: `medmnist` for the data, `timm` for the model,
`scikit-learn` for the metrics, plus `tqdm`/`requests`.

In [1]:
!nvidia-smi

Mon Jul 27 19:49:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 86.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207

In [3]:
!pip install timm medmnist==3.0.2 scikit-learn tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.0 MB/s eta 0:00:00


## Get the survey code

The pipeline lives in this repository's `survey/src/_handlers` package, so the notebook needs a
checkout of it. On Colab it clones into `/content/quantum-quantization`, or `git pull --ff-only`s
that directory if it is already there — so re-running the cell after a push picks up the new code.
Run locally, the notebook already sits inside the repo, so `find_src` climbs to `survey/src` and
git is never touched (your working tree is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `_handlers` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [4]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `_handlers`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / '_handlers').is_dir():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / '_handlers').is_dir(), f'no _handlers package under {SRC}'
print('survey src:', SRC)

survey src: /content/quantum-quantization/survey/src


## Imports

`INFO` comes straight from the `medmnist` package (it carries each dataset's `task` and label
map); `build_dataset` and the training/metric helpers come from the survey's `_handlers`.

`SRC` from the previous cell goes on `sys.path`, and `os.chdir` moves into it so the `./data`
folder the pipeline reads and writes is the shared [`src/data`](../../../data) — the same path the
medmnist `Evaluator` uses inside `evaluation.py`.

In [5]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from medmnist import INFO

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached handler modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m == '_handlers' or m.startswith('_handlers.')]:
    del sys.modules[_m]

from _handlers.datasets import build_dataset
from _handlers.evaluation import (
    build_model,
    train_mnist,
    evaluate_all_metrics,
)

working directory: /content/quantum-quantization/survey/src


## Configuration

`main.py` reads its settings from `argparse` on the command line. Here we replace that block with a
plain config object, so the exact same `args` flows through the pipeline.

**Model** — `convnext_tiny`, timm's port of `facebookresearch/ConvNeXt`'s ConvNeXt-T. The same
family also has `convnext_small`, `convnext_base`, `convnext_large` and `convnext_xlarge` (matching
the paper's T/S/B/L/XL configs), plus `_in22k` variants pretrained on ImageNet-22k for each. With
`pretrained=True` timm downloads the matching ImageNet weights and swaps in a fresh `nb_classes`
head; left at `False` to match the template's train-from-scratch setting.

**Dataset** — any MedMNIST flag: `tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
pneumoniamnist, retinamnist, breastmnist, bloodmnist, organamnist, organcmnist, organsmnist`. The
first download can take a while. The paper's image-folder datasets (`Kvasir`, `CPN`, `Fetal`,
`PAD`, `ISIC2018`) are *not* available without the upstream repo — their downloaders were not
ported.

In [6]:
from types import SimpleNamespace

args = SimpleNamespace(
    model_name='convnext_tiny',                      # any timm ConvNeXt name
    dataset='breastmnist',                           # a medmnist flag
    batch_size=24,
    lr=1e-4,
    epochs=10,
    pretrained=False,                                # load timm's ImageNet weights
    checkpoint_path=None,                            # unused on the timm path
)

## Device

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

Using cuda:0 device.


## Dataset

The `task` field from `INFO` (multi-label vs. multi-class) selects the loss. `build_dataset` then
downloads the `.npz` into `./data`, serves it as 3-channel 224×224, applies the transforms and
returns the datasets plus the number of classes.

In [8]:
# the medmnist `task` selects the loss
info = INFO[args.dataset]
task = info['task']
if task == "multi-label, binary-class":
    loss_function = nn.BCEWithLogitsLoss()
else:
    loss_function = nn.CrossEntropyLoss()

train_dataset, test_dataset, nb_classes = build_dataset(args=args)

print(train_dataset)
print("===================")
print(test_dataset)

Number of channels:  1
Number of classes:  2


100%|██████████| 30.9M/30.9M [00:44<00:00, 702kB/s]


Using downloaded and verified file: ./data/breastmnist_224.npz
Dataset BreastMNIST of size 224 (breastmnist_224)
    Number of datapoints: 546
    Root location: ./data
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'malignant', '1': 'normal, benign'}
    Number of samples: {'train': 546, 'val': 78, 'test': 156}
    Description: The BreastMNIST is based on a dataset of 780 breast ultrasound images. It is categorized into 3 classes: normal, benign, and malignant. As we use low-resolution images, we simplify the task into binary classification by combining normal and benign as positive and classifying them against malignant as negative. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. The source images of 1×500×500 are resized into 1×28×28.
    License: CC BY 4.0
Dataset BreastMNIST of size 224 (breastmnist_224)
    Number of datapoints: 156
    Root location: ./data
    Split: test
    Task: binary-

## Model

`build_model` builds a `MedViT_*` only when the name is a key of `model_classes`; every other name
falls through to `timm.create_model`. Here `model_classes` is **empty**, so `'convnext_tiny'` goes
straight to timm and comes back with an `nb_classes` head, on CUDA. Swapping `args.model_name` for
`convnext_small`/`_base`/`_large` or any other timm name is the only change needed to train a
different backbone.

In [9]:
model_classes = {}   # no MedViT builders — every name falls through to timm

net = build_model(args.model_name, nb_classes, model_classes,
                  pretrained=args.pretrained, checkpoint_path=args.checkpoint_path)

## Optimizer, Scheduler & Data Loaders

AdamW with weight decay and a cosine-annealing schedule stepped **every iteration** — so `T_max` is
the total number of optimizer steps (`epochs * train_num // batch_size`).

In [10]:
train_num = len(train_dataset)
eta = args.epochs * train_num // args.batch_size   # total scheduler steps

optimizer = optim.AdamW(net.parameters(), lr=args.lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)

train_loader = data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*args.batch_size, shuffle=False)

## Train

`train_mnist` (from the handler) runs the loop, scoring every epoch with the medmnist `Evaluator`
(AUC/ACC) and writing the best model to `save_path`.

In [11]:
save_path = f'./{args.model_name}_{args.dataset}.pth'

train_mnist(args.epochs, net, train_loader, test_loader,
            optimizer, scheduler, loss_function, device, save_path, args.dataset, task)

100%|██████████| 4/4 [00:00<00:00,  5.41it/s]
[epoch 1] train_loss: 0.795  auc: 0.468  acc: 0.718

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  5.58it/s]
[epoch 2] train_loss: 0.637  auc: 0.488  acc: 0.712

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  5.52it/s]
[epoch 3] train_loss: 0.601  auc: 0.470  acc: 0.673
100%|██████████| 4/4 [00:00<00:00,  5.45it/s]
[epoch 4] train_loss: 0.632  auc: 0.472  acc: 0.731
100%|██████████| 4/4 [00:00<00:00,  5.39it/s]
[epoch 5] train_loss: 0.585  auc: 0.545  acc: 0.731

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  5.37it/s]
[epoch 6] train_loss: 0.579  auc: 0.622  acc: 0.731

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  5.16it/s]
[epoch 7] train_loss: 0.567  auc: 0.657  acc: 0.731

Saving checkpoint...
100%|██████████| 4/4 [00:00<00:00,  5.23it/s]
[epoch 8] train_loss: 0.581  auc: 0.650  acc: 0.731
100%|██████████| 4/4 [00:00<00:00,  5.31it/s]
[epoch 9] train_loss: 0.559  auc: 0.695  acc: 0.731

Saving 

## Evaluate all metrics

`evaluate_all_metrics` (from the handler) runs the model once over one split and prints **every**
metric the training routines can produce: the medmnist Evaluator AUC/ACC plus accuracy, weighted
precision / recall (sensitivity) / F1, per-class + average specificity, one-vs-rest AUC, the
confusion matrix and a per-class report. Set `split` to `'train'` or `'test'`; uncomment the
`load_state_dict` line to score the best checkpoint instead of the in-memory model.

In [12]:
split = 'test'   # 'train' or 'test'

# net.load_state_dict(torch.load(save_path)['model'])   # uncomment to evaluate the BEST checkpoint

eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(net, eval_dataset, args.dataset, nb_classes, device,
                               split=split, batch_size=2 * args.batch_size)

100%|██████████| 4/4 [00:00<00:00,  4.89it/s]
[medmnist Evaluator]  auc: 0.6967  acc: 0.7308

=== test metrics (2 classes) ===
accuracy            : 0.7308
overall_accuracy    : 0.7308
auc (ovr)           : 0.6967
precision (weighted): 0.5340
recall / sensitivity: 0.7308
specificity (avg)   : 0.5000
f1 (weighted)       : 0.6171

per-class specificity: ['1.000', '0.000']

confusion matrix:
[[  0  42]
 [  0 114]]

classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        42
           1       0.73      1.00      0.84       114

    accuracy                           0.73       156
   macro avg       0.37      0.50      0.42       156
weighted avg       0.53      0.73      0.62       156

